In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# 1. Load the merged dataset from excel
xls = pd.ExcelFile('merged_cleaned.xlsx')
df = pd.read_excel(xls, xls.sheet_names[0])

df

In [66]:
# Copy dataframe
plot_df = df.copy()

# Create Overall Satisfaction
plot_df["Overall Satisfaction"] = (
    plot_df[
        [
            "PRESALES AND PARTNERSHIP",
            "TECHNICAL EXPERTISE",
            "PROJECT DELIVERY",
            "POST-SALES SUPPORT",
        ]
    ].mean(axis=1)
)

plot_df["COGS"] = (
    plot_df["HARDWARE"] + plot_df["SOFTWARE"] + plot_df["MANPOWER"]
)

# Gross Profit and Gross Margin
plot_df["Gross Profit"] = (
    plot_df["REVENUE"]
    - plot_df["COGS"]
)

plot_df["Gross Margin"] = (
    plot_df["Gross Profit"] / plot_df["REVENUE"] * 100
)

# NPS Category
plot_df["NPS Category"] = np.select(
    [
        plot_df["NPS RATING"] >= 9,
        plot_df["NPS RATING"] >= 7
    ],
    [
        "Promoter",
        "Passive"
    ],
    default="Detractor"
)

# Add small horizontal jitter
np.random.seed(42)

plot_df["Satisfaction Jitter"] = (
    plot_df["Overall Satisfaction"]
    + np.random.uniform(-0.1, 0.1, len(plot_df))
).clip(lower=1, upper=5)


## NPS Category Margin Analysis

Assess whether profitability differs by NPS Category and translate the observed margin patterns into appropriate account-management actions.

The analysis uses the same record-level Gross Margin values shown in the preceding box plot. It compares each category's median, interquartile range (IQR), standard deviation, and share of records with a positive gross margin. A category is described as producing *consistently higher* margins only when both its median and its 25th percentile exceed those of the other NPS categories.

Run the cell below to produce the statistical summary and evidence-based management recommendations.


In [68]:
hover_fields = ["REVENUE", "Gross Profit","Overall Satisfaction"]
    

fig = px.ecdf(
    plot_df,
    x="Gross Margin",
    color="NPS Category"
)

fig.update_traces(
    # boxmean=True,  # Show mean as a line
    marker={"size": 6, "opacity": 0.8, "line": {"width": 0.5, "color": "white"}}
)

fig.update_layout(
    yaxis_title="Gross Margin (%)",
    width=600, height=1000
)

fig.show()

## Revenue and Margin Treemap

### Objective
Identify client-type and NPS segments that combine meaningful revenue contribution with healthy gross margins.

### Method
Tile size represents aggregated revenue, while colour represents gross margin. This separates commercial scale from profitability: a large red tile requires attention because it contributes substantial revenue but has a lower gross margin.

### Results
Use the hierarchy to compare Client Type first, then NPS Category. Hover over a tile for revenue, gross profit, gross margin, number of clients, and number of records.


In [ ]:
# Aggregate to the displayed treemap hierarchy so revenue and margin are interpreted at segment level.
treemap_df = (
    plot_df.groupby(["TYPE", "NPS Category"], observed=True)
    .agg(
        Revenue=("REVENUE", "sum"),
        Gross_Profit=("Gross Profit", "sum"),
        Client_Count=("Client ID", "nunique"),
        Record_Count=("Client ID", "size"),
    )
    .reset_index()
    .rename(columns={"TYPE": "Client Type"})
)
treemap_df["Gross Margin"] = (
    treemap_df["Gross_Profit"] / treemap_df["Revenue"] * 100
)

overall_margin = plot_df["Gross Margin"].median()

fig = px.treemap(
    treemap_df,
    path=["Client Type", "NPS Category"],
    values="Revenue",
    color="Gross Margin",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=overall_margin,
    custom_data=["Revenue", "Gross_Profit", "Gross Margin", "Client_Count", "Record_Count"],
    template="plotly_white",
    title="Revenue Contribution and Gross Margin by Client Type and NPS Category",
)

fig.update_traces(
    texttemplate="<b>%{label}</b><br>$%{value:,.0f}",
    textfont={"size": 15},
    marker={"line": {"color": "white", "width": 2}},
    hovertemplate=(
        "<b>%{label}</b><br>"
        "<b>Revenue:</b> $%{customdata[0]:,.0f}<br>"
        "<b>Gross Profit:</b> $%{customdata[1]:,.0f}<br>"
        "<b>Gross Margin:</b> %{customdata[2]:.1f}%<br>"
        "<b>Clients:</b> %{customdata[3]:,.0f}<br>"
        "<b>Records:</b> %{customdata[4]:,.0f}<extra></extra>"
    ),
)

fig.update_layout(
    title={"x": 0.5, "xanchor": "center", "font": {"size": 22}},
    font={"family": "Arial", "size": 13},
    coloraxis_colorbar={
        "title": "Gross Margin (%)",
        "ticksuffix": "%",
        "len": 0.75,
    },
    width=1150,
    height=700,
    margin={"l": 25, "r": 105, "t": 80, "b": 25},
)
fig.show()


## Statistical observations
- **Median comparison:** Promoter median margin is **42.6%**; Passive and Detractor medians are shown in the table for direct comparison. The median is preferred for category comparison because it is less sensitive to outliers than the mean.
- **Spread comparison:** **Detractor** has the widest middle 50% of margins, with an IQR of **16.6 percentage points**. A wider IQR indicates less predictable profitability within that category.
- **Promoter consistency:** Promoters do not consistently produce higher margins under the stated median-and-lower-quartile test; a higher average or median alone should not be interpreted as a consistent margin advantage.
- **Detractor profitability:** Detractors remain profitable on a typical basis: their median margin is 46.8% and 100.0% of detractor records have a positive margin.

## Management actions by NPS Category
- **Promoters:** Protect service quality, identify advocacy and referral opportunities, and pursue expansion only where the account continues to meet margin expectations.
- **Passives:** Use proactive account reviews to identify friction points and move them toward promoter status. Prioritise actions that lift satisfaction without increasing cost-to-serve disproportionately.
- **Detractors:** Triage accounts by both satisfaction and margin. Recover high-margin detractor relationships through senior outreach and root-cause resolution; for persistently low-margin detractors, reprice, reduce cost-to-serve, or reconsider the relationship.

## Limitation
These results describe association between NPS Category and Gross Margin in the available records. They do not establish that NPS causes margin differences.

### Financial aggregation validation

The analysis now loads **`merged_cleaned.xlsx`**, which was created from a validated one-to-one merge between survey and COGS records on **Client ID + Year**, followed by a one-to-many profile merge on **Client ID**. The cleaning workflow reports zero duplicate Client ID–Year records after the merge. Therefore, financial values are aggregated from a canonical client-year grain before being rolled up to client, NPS, or priority-group level.

**Control:** Before reporting updated totals, rerun the notebook from the first cell so every visual uses the same validated cleaned dataset.


## Client Priority Matrix

Classify each client by profitability and satisfaction to identify where account-management effort should be directed.

The chart aggregates the available records to one point per client. Overall Satisfaction is the mean of the four service ratings, while Gross Margin is calculated as total Gross Profit divided by total Revenue. Median reference lines create four action-oriented quadrants:

- **Prioritise** — below-median satisfaction and above-median gross margin
- **Retain** — above-median satisfaction and above-median gross margin
- **Improve** — above-median satisfaction and below-median gross margin
- **Reconsider** — below-median satisfaction and below-median gross margin


In [ ]:
from IPython.display import Markdown, display

# Aggregate repeated yearly records so that each marker represents one client.
client_summary = (
    plot_df.groupby("Client ID", as_index=False)
    .agg(
        {
            "TYPE": "first",
            "Overall Satisfaction": "mean",
            "REVENUE": "sum",
            "Gross Profit": "sum",
            "NPS RATING": "mean",
        }
    )
    .rename(columns={"TYPE": "Client Type"})
)

# Recalculate margin from client-level totals so large and small annual records are weighted correctly.
client_summary["Gross Margin"] = client_summary["Gross Profit"] / client_summary["REVENUE"] * 100
client_summary["NPS Category"] = np.select([client_summary["NPS RATING"] >= 9,client_summary["NPS RATING"] >= 7,],["Promoter", "Passive"],default="Detractor")

median_satisfaction = client_summary["Overall Satisfaction"].median()
median_gross_margin = client_summary["Gross Margin"].median()

priority_conditions = [
    (client_summary["Overall Satisfaction"] < median_satisfaction)
    & (client_summary["Gross Margin"] >= median_gross_margin),
    (client_summary["Overall Satisfaction"] >= median_satisfaction)
    & (client_summary["Gross Margin"] >= median_gross_margin),
    (client_summary["Overall Satisfaction"] >= median_satisfaction)
    & (client_summary["Gross Margin"] < median_gross_margin),
]
client_summary["Priority Group"] = np.select(priority_conditions, ["Prioritise", "Retain", "Improve"], default="Reconsider")

priority_order = ["Prioritise", "Retain", "Improve", "Reconsider"]
priority_summary = (
client_summary.groupby("Priority Group", observed=True)
    .agg(
        Clients=("Client ID", "nunique"),
        Revenue=("REVENUE", "sum"),
        Gross_Profit=("Gross Profit", "sum"),
        Average_Gross_Margin=("Gross Margin", "mean"),
    )
    .reindex(priority_order, fill_value=0)
)

hover_fields = [
    "Client ID",
    "Client Type",
    "REVENUE",
    "Gross Profit",
    "Gross Margin",
    "Overall Satisfaction",
    "NPS Category",
    "Priority Group",
]

fig = px.scatter(
    client_summary,
    x="Overall Satisfaction",
    y="Gross Margin",
    color="Client Type",
    custom_data=hover_fields,
    template="plotly_white",
    title="Client Priority Matrix: Satisfaction and Gross Margin",
)

for trace in fig.data:
    trace.update(
        marker={"size": 12, "opacity": 0.78, "line": {"width": 0.7, "color": "white"}},
        hovertemplate=(
            "<b>Client ID:</b> %{customdata[0]}<br>"
            "<b>Client Type:</b> %{customdata[1]}<br>"
            "<b>Revenue:</b> $%{customdata[2]:,.0f}<br>"
            "<b>Gross Profit:</b> $%{customdata[3]:,.0f}<br>"
            "<b>Gross Margin:</b> %{customdata[4]:.1f}%<br>"
            "<b>Overall Satisfaction:</b> %{customdata[5]:.2f}<br>"
            "<b>NPS Category:</b> %{customdata[6]}<br>"
            "<b>Priority Group:</b> %{customdata[7]}<extra></extra>"
        ),
    )

fig.add_vline(
    x=median_satisfaction,
    line_dash="dash",
    line_color="#4A5568",
    annotation_text=f"Median satisfaction: {median_satisfaction:.2f}",
    annotation_position="top right",
)
fig.add_hline(
    y=median_gross_margin,
    line_dash="dash",
    line_color="#4A5568",
    annotation_text=f"Median gross margin: {median_gross_margin:.1f}%",
    annotation_position="bottom right",
)

x_min, x_max = client_summary["Overall Satisfaction"].min(), client_summary["Overall Satisfaction"].max()
y_min, y_max = client_summary["Gross Margin"].min(), client_summary["Gross Margin"].max()
quadrant_labels = [
    ("Prioritise", (x_min + median_satisfaction) / 2, (median_gross_margin + y_max) / 2, "red"),
    ("Retain", (median_satisfaction + x_max) / 2, (median_gross_margin + y_max) / 2, "green"),
    ("Reconsider", (x_min + median_satisfaction) / 2, (y_min + median_gross_margin) / 2, "brown"),
    ("Improve", (median_satisfaction + x_max) / 2, (y_min + median_gross_margin) / 2, "blue"),
]
for label, x_position, y_position, colour in quadrant_labels:
    fig.add_annotation(
        x=x_position,
        y=y_position,
        text=f"<b>{label}</b>",
        showarrow=False,
        font={"size": 16, "color": colour},
        bgcolor="rgba(255, 255, 255, 0.82)",
        bordercolor=colour,
        borderwidth=1,
        borderpad=6,
    )

fig.update_layout(
    title={"x": 0.5, "xanchor": "center", "font": {"size": 23}},
    xaxis_title="Overall Satisfaction",
    yaxis_title="Gross Margin (%)",
    legend_title="Client Type",
    hoverlabel={"bgcolor": "white", "font_size": 13, "font_family": "Arial"},
    font={"family": "Arial", "size": 13},
    width=1150,
    height=720,
    margin={"l": 80, "r": 40, "t": 85, "b": 75},
)
fig.update_xaxes(showgrid=True, gridcolor="#E5E7EB", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="#E5E7EB", zeroline=False, ticksuffix="%")
fig.show()

quadrant_counts = ", ".join(
    f"{group}: {int(priority_summary.loc[group, 'Clients'])}"
    for group in priority_order
)
prioritise_clients = int(priority_summary.loc["Prioritise", "Clients"])
prioritise_revenue = priority_summary.loc["Prioritise", "Revenue"]
prioritise_gross_profit = priority_summary.loc["Prioritise", "Gross_Profit"]

# display(Markdown(
#     f'''## Observations
# - The matrix contains **{client_summary['Client ID'].nunique():,} clients**, with one aggregated point per client.
# - The median benchmarks are **{median_satisfaction:.2f} / 5** for Overall Satisfaction and **{median_gross_margin:.1f}%** for Gross Margin.
# - Client distribution by priority group: **{quadrant_counts}**.
# - The **Prioritise** quadrant contains **{prioritise_clients:,} clients**, representing **${prioritise_revenue:,.0f}** in revenue and **${prioritise_gross_profit:,.0f}** in gross profit.

# ## Business interpretation
# The four quadrants distinguish customer experience from financial contribution. **Retain** clients combine strong satisfaction and margin, while **Prioritise** clients are currently profitable but have below-median satisfaction. **Improve** clients are satisfied but have lower margins, suggesting a need to review cost-to-serve or account expansion opportunities. **Reconsider** clients have both lower satisfaction and lower margin, so interventions should be selective and commercially justified.

# ## Highest strategic attention
# **Prioritise** deserves the highest strategic attention because these clients generate above-median margins but show below-median satisfaction. Targeted account recovery can protect valuable gross profit before dissatisfaction escalates into churn or reduced spending.

# ## Research-question response
# The matrix answers the research question by identifying which individual clients require different strategic actions based on the combined evidence of satisfaction and profitability. It enables account teams to focus retention resources on valuable at-risk clients rather than applying the same treatment to every client.
# '''
# ))


### Observations
The matrix contains 140 clients, with one aggregated point per client.
The median benchmarks are 4.00 / 5 for Overall Satisfaction and 43.9% for Gross Margin.
Client distribution by priority group: Prioritise: 29, Retain: 41, Improve: 37, Reconsider: 33.
The Prioritise quadrant contains 29 clients, representing $91,815,141 in revenue and $45,424,623 in gross profit.

### Business interpretation
The four quadrants compare customer experience to financial contribution. Retain clients have strong satisfaction and margin, while Prioritise clients are currently profitable but have below-median satisfaction. Improve clients are satisfied but have lower margins, suggesting a need to review cost-to-serve or account expansion opportunities. Reconsider clients have both lower satisfaction and lower margin, so interventions should be selective and commercially justified.

### Highest strategic attention
Prioritise deserves the highest strategic attention because these clients generate above-median margins but show below-median satisfaction. Targeted account recovery can protect valuable gross profit before dissatisfaction escalates into churn or reduced spending.

### Research-question response
The matrix answers the research question by identifying which individual clients require different strategic actions based on the combined evidence of satisfaction and profitability. It enables account teams to focus retention resources on valuable at-risk clients rather than applying the same treatment to every client.

## Portfolio Resource Allocation Sunburst

### Objective
Show how revenue is distributed from Client Type to NPS Category and then to Priority Quadrant, so portfolio resources can be directed to the most commercially significant client segments.

### Method
The sunburst uses the existing client-level classification from the priority matrix. Segment area represents aggregated **Revenue**, and the hierarchy is:

**Client Type → NPS Category → Priority Quadrant**

Hover over a segment to see revenue, gross profit, gross margin, and the number of unique clients.

### Resource allocation guide
- **Prioritise:** allocate senior account-management and service-recovery capacity first, because these clients are commercially valuable but experience risk.
- **Retain:** protect service quality and identify efficient growth opportunities.
- **Improve:** review cost-to-serve and margin-improvement opportunities while maintaining satisfaction.
- **Reconsider:** apply selective intervention; reprice, redesign service, or exit where recovery is not commercially justified.

In [72]:
# Aggregate the existing client-level classifications so segment area represents portfolio revenue.
sunburst_df = (
    client_summary.groupby(
        ["Client Type", "NPS Category", "Priority Group"],
        observed=True,
    )
    .agg(
        Revenue=("REVENUE", "sum"),
        Gross_Profit=("Gross Profit", "sum"),
        Client_Count=("Client ID", "nunique"),
    )
    .reset_index()
)
sunburst_df["Gross Margin"] = (
    sunburst_df["Gross_Profit"] / sunburst_df["Revenue"] * 100
)

fig = px.sunburst(
    sunburst_df,
    path=["Client Type", "NPS Category", "Priority Group"],
    values="Revenue",
    color="Gross Margin",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=client_summary["Gross Margin"].median(),
    custom_data=["Revenue", "Gross_Profit", "Gross Margin", "Client_Count"],
    template="plotly_white",
    title="Portfolio Resource Allocation: Revenue by Client Type, NPS Category, and Priority Quadrant",
)

fig.update_traces(
    textinfo="label+percent parent",
    insidetextorientation="radial",
    marker={"line": {"color": "white", "width": 2}},
    hovertemplate=(
        "<b>%{label}</b><br>"
        "<b>Revenue:</b> $%{customdata[0]:,.0f}<br>"
        "<b>Gross Profit:</b> $%{customdata[1]:,.0f}<br>"
        "<b>Gross Margin:</b> %{customdata[2]:.1f}%<br>"
        "<b>Unique Clients:</b> %{customdata[3]:,.0f}"
        "<extra></extra>"
    ),
)

fig.update_layout(
    title={"x": 0.5, "xanchor": "center", "font": {"size": 22}},
    font={"family": "Arial", "size": 13},
    coloraxis_colorbar={
        "title": "Gross Margin (%)",
        "ticksuffix": "%",
        "len": 0.75,
    },
    width=1150,
    height=720,
    margin={"l": 20, "r": 110, "t": 90, "b": 20},
)
fig.show()

priority_revenue = (
    sunburst_df.groupby("Priority Group", observed=True)["Revenue"]
    .sum()
    .reindex(["Prioritise", "Retain", "Improve", "Reconsider"], fill_value=0)
)
print("Revenue by priority quadrant:")
display(priority_revenue.to_frame("Revenue").style.format("${:,.0f}"))

Revenue by priority quadrant:


,Revenue
Priority Group,
Prioritise,"$91,815,141"
Retain,"$182,377,830"
Improve,"$153,533,609"
Reconsider,"$113,120,067"
